# EmpowerLens — Kaggle GPU runner

Runs a real transformer fine-tune on Kaggle's free GPU, then makes `results/` downloadable.

**Before running:**
1. Settings → **Accelerator: GPU**, and **Internet: On** (needed for `pip` + the roberta-base download).
2. The frozen splits in `data/splits/` must already be **committed and pushed** on the branch below — the notebook uses them as-is and never regenerates them.

This notebook only orchestrates shell commands; all logic lives in `src/`.

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt

In [ ]:
# 2. Choose ONE task, train it over the three seeds, evaluate each on val+test,
#    then aggregate. --device auto resolves to the Kaggle GPU (cuda).
TASK = "multiclass"  # one of: binary | multiclass | multilabel

for seed in (42, 1337, 2024):
    ckpt = f"checkpoints/{TASK}_roberta-base_{seed}"
    !python -m src.train_transformer --task $TASK --seed $seed --device auto
    !python -m src.evaluate --checkpoint $ckpt --reference

!python -m src.aggregate

In [ ]:
# 3. Copy results/ to the Kaggle output so it can be downloaded from the session.
!mkdir -p /kaggle/working/results
!cp -r results/* /kaggle/working/results/
!ls -la /kaggle/working/results